# Water Quality — CatBoost Tabular Baseline

**Feature sources**
| Source | What | Notes |
|--------|------|-------|
| Landsat | NDMI, MNDWI | Surface-water spectral indices |
| ERA5 daily | Aggregate windows (1 d / 7 d / 15 d / 30 d) | Precip, temp, dewpoint, pressure |
| SoilGrids | Physical soil properties (0–100 cm) | Drop sand & >1 m layers |
| Calendar | month, day_of_year | Seasonality |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor, Pool

BASE    = Path('.')
TARGETS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
N_FOLDS = 5
SEED    = 42

# ERA5 variables to keep (wind speed / MSLP dropped — not informative for water quality)
ERA5_VARS = [
    'dewpoint_2m_temperature',
    'maximum_2m_air_temperature',
    'mean_2m_air_temperature',
    'minimum_2m_air_temperature',
    'surface_pressure',
    'total_precipitation',
]

# Aggregation windows (days) computed from the 30-day lookback
LAG_WINDOWS = [1, 7, 15, 30]

# ── Feature toggles ───────────────────────────────────────────────────────────
USE_SOIL     = True   # SoilGrids features (bdod, cec, clay, phh2o, silt, soc @ 0–100 cm)
USE_LANDSAT  = True   # Landsat spectral indices (NDMI, MNDWI)
USE_ERA5     = True   # ERA5 aggregate lag windows
USE_CALENDAR = True   # month, day_of_year

# ── GPU / device ──────────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    DEVICE = 'cuda'
    for i in range(torch.cuda.device_count()):
        mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}  ({mem:.0f} GB)')
else:
    DEVICE = 'cpu'
    print('  No GPU detected — running on CPU')

print('Libraries loaded  |  device:', DEVICE)

## 1. Load raw files

In [ ]:
train_raw  = pd.read_csv(BASE / 'data/water_quality_training_dataset.csv')
sub_raw    = pd.read_csv(BASE / 'data/submission_template.csv')
soil_raw   = pd.read_csv(BASE / 'data/soilgrids_features_full.csv')
era5_lb    = pd.read_csv(BASE / 'data/era5/era5_lookback_30d.csv')
ls_train   = pd.read_csv(BASE / 'data/landsat_features_training_ORIGINAL.csv')
ls_val     = pd.read_csv(BASE / 'data/landsat_features_validation_ORIGINAL.csv')

print(f'train: {train_raw.shape}  sub: {sub_raw.shape}')
print(f'soilgrids: {soil_raw.shape}')
print(f'era5 lookback: {era5_lb.shape}')
print(f'landsat train: {ls_train.shape}  landsat val: {ls_val.shape}')

## 2. Build feature matrix

In [ ]:
# ── Stack train + submission ──────────────────────────────────────────────────
train_raw['source'] = 'train'
sub_raw['source']   = 'submission'
merged = pd.concat([train_raw, sub_raw], ignore_index=True)
merged['sample_date'] = pd.to_datetime(merged['Sample Date'], dayfirst=True, errors='coerce')
merged['lat_r6'] = merged['Latitude'].round(6)
merged['lon_r6'] = merged['Longitude'].round(6)

KEY = ['lat_r6', 'lon_r6', 'sample_date']

# ── NDMI / MNDWI from Landsat ─────────────────────────────────────────────────
ls = pd.concat([ls_train, ls_val], ignore_index=True)
ls['sample_date'] = pd.to_datetime(ls['Sample Date'], dayfirst=True, errors='coerce')
ls['lat_r6'] = ls['Latitude'].round(6)
ls['lon_r6'] = ls['Longitude'].round(6)
ls_key = ls[['lat_r6', 'lon_r6', 'sample_date', 'NDMI', 'MNDWI']].drop_duplicates(KEY)
merged = merged.merge(ls_key, on=KEY, how='left')

print('After landsat merge — NDMI NaN rate:', merged['NDMI'].isna().mean().round(3))

### 2a. ERA5 aggregate lag features

In [ ]:
era5_lb['sample_date'] = pd.to_datetime(era5_lb['Sample Date'], dayfirst=True, errors='coerce')
era5_lb['lat_r6'] = pd.to_numeric(era5_lb['Latitude'], errors='coerce').round(6)
era5_lb['lon_r6'] = pd.to_numeric(era5_lb['Longitude'], errors='coerce').round(6)

# Compute rolling-window aggregates from the per-day lag columns
era5_agg_cols = []
for var in ERA5_VARS:
    for w in LAG_WINDOWS:
        lag_cols = [f'{var}_lag{i:02d}' for i in range(1, w + 1)]
        available = [c for c in lag_cols if c in era5_lb.columns]
        if not available:
            continue
        feat_name = f'{var}_{w}d'
        era5_lb[feat_name] = era5_lb[available].mean(axis=1)
        era5_agg_cols.append(feat_name)

era5_key = era5_lb[KEY + era5_agg_cols].drop_duplicates(KEY)
merged   = merged.merge(era5_key, on=KEY, how='left')

print(f'ERA5 aggregate features: {len(era5_agg_cols)}')
print('NaN rate (top 5):', merged[era5_agg_cols].isna().mean().sort_values(ascending=False).head())

### 2b. SoilGrids features (no sand, max depth 100 cm)

In [ ]:
SOIL_KEEP_PROPS   = ['bdod', 'cec', 'clay', 'phh2o', 'silt', 'soc']   # sand dropped
SOIL_KEEP_DEPTHS  = ['0_5cm', '5_15cm', '15_30cm', '30_60cm', '60_100cm']  # 100_200cm dropped

SOIL_COLS = [
    f'soil_{prop}_{depth}'
    for prop  in SOIL_KEEP_PROPS
    for depth in SOIL_KEEP_DEPTHS
    if f'soil_{prop}_{depth}' in soil_raw.columns
]

soil_key = soil_raw.rename(columns={'lat': 'lat_r6', 'lon': 'lon_r6'}).copy()
soil_key['lat_r6'] = soil_key['lat_r6'].round(6)
soil_key['lon_r6'] = soil_key['lon_r6'].round(6)
soil_key = soil_key[['lat_r6', 'lon_r6'] + SOIL_COLS].drop_duplicates(['lat_r6', 'lon_r6'])
merged   = merged.merge(soil_key, on=['lat_r6', 'lon_r6'], how='left')

print(f'Soil features: {len(SOIL_COLS)}')
print('NaN rate (soil):', merged[SOIL_COLS].isna().mean().max().round(3))

### 2c. Calendar features & final feature list

In [ ]:
merged['month']      = merged['sample_date'].dt.month
merged['day_of_year'] = merged['sample_date'].dt.dayofyear

LANDSAT_COLS  = ['NDMI', 'MNDWI']
CALENDAR_COLS = ['month', 'day_of_year']

ALL_FEATURE_COLS = (
    (LANDSAT_COLS  if USE_LANDSAT  else []) +
    (era5_agg_cols if USE_ERA5     else []) +
    (SOIL_COLS     if USE_SOIL     else []) +
    (CALENDAR_COLS if USE_CALENDAR else [])
)
ALL_FEATURE_COLS = [c for c in ALL_FEATURE_COLS if c in merged.columns]

print(f'Total features : {len(ALL_FEATURE_COLS)}')
print(f'  Landsat      : {len(LANDSAT_COLS)  if USE_LANDSAT  else 0} (USE_LANDSAT={USE_LANDSAT})')
print(f'  ERA5 lags    : {len(era5_agg_cols) if USE_ERA5     else 0} (USE_ERA5={USE_ERA5})')
print(f'  Soil         : {len(SOIL_COLS)     if USE_SOIL     else 0} (USE_SOIL={USE_SOIL})')
print(f'  Calendar     : {len(CALENDAR_COLS) if USE_CALENDAR else 0} (USE_CALENDAR={USE_CALENDAR})')
nan_summary = merged[ALL_FEATURE_COLS].isna().mean().sort_values(ascending=False)
print('\nTop NaN rates:')
print(nan_summary.head(10).to_string())

## 3. Split train / submission

In [ ]:
train_df = merged[merged['source'] == 'train'].copy().reset_index(drop=True)
sub_df   = merged[merged['source'] == 'submission'].copy().reset_index(drop=True)

# Group key for GroupKFold — avoids spatial leakage across same location
train_df['location_group'] = (
    train_df['Latitude'].round(3).astype(str) + '_' +
    train_df['Longitude'].round(3).astype(str)
)

print(f'Train: {len(train_df)}  |  Submission: {len(sub_df)}')
print('Targets available:', {t: int(train_df[t].notna().sum()) for t in TARGETS})

## 4. Hyperparameter tuning (optional)

Run this cell to search for better params via Optuna.  
The result is stored as `BEST_PARAMS` and picked up by the CV cell automatically.

In [ ]:
from tuning import run_optuna_study

TUNE_TARGET = TARGETS[0]
mask_opt    = train_df[TUNE_TARGET].notna()

BEST_PARAMS, study = run_optuna_study(
    X          = train_df.loc[mask_opt, ALL_FEATURE_COLS],
    y          = train_df.loc[mask_opt, TUNE_TARGET],
    groups     = train_df.loc[mask_opt, 'location_group'],
    seed       = SEED,
    n_trials   = 30,
    n_folds    = 3,
    storage_path = str(BASE / 'data/optuna.db'),
    study_name = f'catboost_{TUNE_TARGET.replace(" ", "_")}',
)

print(f'Best R² ({TUNE_TARGET}): {study.best_value:.4f}')
print('Best params:', BEST_PARAMS)

## 5. Location-aware K-Fold CV

In [ ]:
# Fall back to sensible defaults if tuning cell was skipped
_base_params = dict(
    iterations        = 2000,
    learning_rate     = 0.07,
    depth             = 6,
    l2_leaf_reg       = 1.5,
    subsample         = 0.8,
    colsample_bylevel = 0.5,
    loss_function     = 'RMSE',
    eval_metric       = 'RMSE',
    random_seed       = SEED,
    od_type           = 'Iter',
    od_wait           = 100,
    verbose           = False,
    allow_writing_files = False,
)

try:
    _base_params.update(BEST_PARAMS)
    print('Using tuned params:', BEST_PARAMS)
except NameError:
    print('Using default params  (run section 4 to override)')

gkf = GroupKFold(n_splits=N_FOLDS)

cv_results = {
    t: {'oof': np.full(len(train_df), np.nan), 'r2': [], 'rmse': [], 'models': [], 'best_iters': []}
    for t in TARGETS
}

for target in TARGETS:
    mask   = train_df[target].notna()
    X_all  = train_df.loc[mask, ALL_FEATURE_COLS]
    y_all  = train_df.loc[mask, target]
    groups = train_df.loc[mask, 'location_group']
    idxs   = train_df.index[mask]

    print(f'\n=== {target}  (n={len(y_all)}) ===')

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_all, y_all, groups)):
        model = CatBoostRegressor(**_base_params)
        model.fit(Pool(X_all.iloc[tr_idx], y_all.iloc[tr_idx]),
                  eval_set=Pool(X_all.iloc[va_idx], y_all.iloc[va_idx]))

        preds = model.predict(X_all.iloc[va_idx])
        r2    = r2_score(y_all.iloc[va_idx], preds)
        rmse  = np.sqrt(np.mean((y_all.iloc[va_idx] - preds) ** 2))

        cv_results[target]['oof'][idxs[va_idx]] = preds
        cv_results[target]['r2'].append(r2)
        cv_results[target]['rmse'].append(rmse)
        cv_results[target]['models'].append(model)
        cv_results[target]['best_iters'].append(model.best_iteration_)

        print(f'  Fold {fold+1}: R²={r2:.4f}  RMSE={rmse:.3f}  iter={model.best_iteration_}')

    print(f'  MEAN : R²={np.mean(cv_results[target]["r2"]):.4f}  '
          f'RMSE={np.mean(cv_results[target]["rmse"]):.3f}  '
          f'avg_iter={int(np.mean(cv_results[target]["best_iters"]))}')

## 6. CV Summary

In [ ]:
rows = [
    {
        'Target':     t,
        'Mean R²':    np.mean(cv_results[t]['r2']),
        'Std R²':     np.std(cv_results[t]['r2']),
        'Mean RMSE':  np.mean(cv_results[t]['rmse']),
        'Std RMSE':   np.std(cv_results[t]['rmse']),
    }
    for t in TARGETS
]
pd.DataFrame(rows).set_index('Target').round(4)

## 7. Feature Importance

## 7. Alternative models — TabPFN & TabM

Both models predict **all 3 targets in one pass** (multi-output).
Instead of K-fold, a single location-aware 80/20 split is used —
K-fold on top of an internal ensemble would be double-stacking.

| Model | Strategy | Notes |
|-------|----------|-------|
| **TabPFN v2** | Single 80/20 val split | No training; 80 % of rows = context; 47 GB VRAM handles full context |
| **TabM k=32** | Single 80/20 val split + early stopping | Multi-output `d_out=3`; NaN-masked loss per target; `torch.compile` on A6000 |

### 7a. TabPFN

TabPFN is a **pretrained transformer foundation model** — zero training, zero hyperparameters.
It uses in-context learning: the training rows are the model's "context" at inference time.

Multi-output is supported natively: pass a `(N, 3)` target matrix and get back `(N, 3)` predictions
in a single forward pass.

> To upgrade from v2 → v2.5 (better weights): run `hf auth login` in the terminal,
> then change `ModelVersion.V2` to `ModelVersion.V2_5` below.

In [ ]:
from tabpfn import TabPFNRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit

# TabPFN requires 1-D targets — MultiOutputRegressor fits one model per target
# but exposes a single .fit(X, Y) / .predict(X) -> (N, 3) interface
imputer_pfn = SimpleImputer(strategy="median")
X_pfn = pd.DataFrame(
    imputer_pfn.fit_transform(train_df[ALL_FEATURE_COLS]),
    columns=ALL_FEATURE_COLS,
)
Y_pfn = train_df[TARGETS].values.astype(np.float32)  # (N, 3)
groups_pfn = train_df["location_group"]

# Single location-aware 80/20 split (reused by TabM cell below)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
tr_idx, va_idx = next(gss.split(X_pfn, Y_pfn, groups_pfn))

# MultiOutputRegressor wraps TabPFN — one model per target,
# single .fit / .predict call from the outside
pfn_model = MultiOutputRegressor(
    TabPFNRegressor(device=DEVICE),
    n_jobs=1,  # TabPFN is already GPU-accelerated
)
pfn_model.fit(X_pfn.iloc[tr_idx], Y_pfn[tr_idx])
pfn_preds = pfn_model.predict(X_pfn.iloc[va_idx])  # (N_val, 3)

tabpfn_val_metrics = {}
for i, target in enumerate(TARGETS):
    r2   = r2_score(Y_pfn[va_idx, i], pfn_preds[:, i])
    rmse = np.sqrt(np.mean((Y_pfn[va_idx, i] - pfn_preds[:, i]) ** 2))
    tabpfn_val_metrics[target] = {"r2": r2, "rmse": rmse}
    print(f"TabPFN  {target:<40s}  R²={r2:.4f}  RMSE={rmse:.3f}")


### 7b. TabM

TabM (ICLR 2025) trains `k=32` MLP sub-models with shared weights (BatchEnsemble).
With `d_out=3` all targets are predicted simultaneously — no separate runs per target.

**Loss**: per-target MSE averaged over non-NaN targets (robust to missing labels).
**Inference**: average the `k` predictions → `(B, 3)`.  
**Early stopping**: monitored on the same 80/20 val split used for TabPFN.

In [ ]:
import tabm as tabm_lib
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

TABM_EPOCHS     = 300
TABM_BATCH_SIZE = 512
TABM_LR         = 2e-3      # recommended default
TABM_WD         = 3e-4      # recommended default
TABM_K          = 32        # recommended default ensemble size
TABM_PATIENCE   = 30        # early-stopping patience (epochs without improvement)

# ── Prepare data (same 80/20 split as TabPFN) ────────────────────────────────
imputer_tabm = SimpleImputer(strategy='median')
X_tabm = imputer_tabm.fit_transform(train_df[ALL_FEATURE_COLS]).astype(np.float32)
Y_tabm = train_df[TARGETS].values.astype(np.float32)  # (N, 3); NaN allowed

# tr_idx / va_idx reuse the GroupShuffleSplit from the TabPFN cell
X_tr_t = torch.tensor(X_tabm[tr_idx], device=DEVICE)
Y_tr_t = torch.tensor(Y_tabm[tr_idx], device=DEVICE)  # may contain NaN
X_va_t = torch.tensor(X_tabm[va_idx], device=DEVICE)
Y_va_np = Y_tabm[va_idx]  # keep as numpy for metric computation

# ── Build model ──────────────────────────────────────────────────────────────
n_features = X_tabm.shape[1]
n_targets  = len(TARGETS)

tabm_model = tabm_lib.TabM.make(n_num_features=n_features, d_out=n_targets).to(DEVICE)
if DEVICE != 'cpu':
    tabm_model = torch.compile(tabm_model)

opt = torch.optim.AdamW(tabm_model.parameters(), lr=TABM_LR, weight_decay=TABM_WD)

loader = DataLoader(
    TensorDataset(X_tr_t, Y_tr_t),
    batch_size=TABM_BATCH_SIZE, shuffle=True, drop_last=False,
)

# ── Training loop ────────────────────────────────────────────────────────────
best_val_rmse = float('inf')
best_state    = None
patience_cnt  = 0

for epoch in range(TABM_EPOCHS):
    tabm_model.train()
    for xb, yb in loader:
        opt.zero_grad()
        pred = tabm_model(xb)          # (B, k, n_targets)
        # Per-target MSE with NaN masking; train k predictions independently
        loss = torch.zeros(1, device=DEVICE)
        for t_i in range(n_targets):
            valid = ~torch.isnan(yb[:, t_i])
            if valid.sum() == 0:
                continue
            p = pred[valid, :, t_i]                        # (N_ok, k)
            y = yb[valid, t_i].unsqueeze(1).expand_as(p)  # (N_ok, k)
            loss = loss + F.mse_loss(p, y)
        (loss / n_targets).backward()
        opt.step()

    # Validation: average k predictions → (N_val, n_targets)
    tabm_model.eval()
    with torch.no_grad():
        val_pred = tabm_model(X_va_t).mean(dim=1).cpu().numpy()  # (N_val, n_targets)

    # Average RMSE over non-NaN targets
    rmses = []
    for t_i in range(n_targets):
        ok = ~np.isnan(Y_va_np[:, t_i])
        if ok.sum():
            rmses.append(np.sqrt(np.mean((Y_va_np[ok, t_i] - val_pred[ok, t_i])**2)))
    val_rmse = np.mean(rmses)

    if val_rmse < best_val_rmse - 1e-6:
        best_val_rmse = val_rmse
        best_state    = {k: v.cpu().clone() for k, v in tabm_model.state_dict().items()}
        patience_cnt  = 0
    else:
        patience_cnt += 1
        if patience_cnt >= TABM_PATIENCE:
            print(f'Early stop at epoch {epoch+1}  best_val_rmse={best_val_rmse:.3f}')
            break

tabm_model.load_state_dict(best_state)
tabm_model.eval()
with torch.no_grad():
    tabm_preds = tabm_model(X_va_t).mean(dim=1).cpu().numpy()  # (N_val, n_targets)

tabm_val_metrics = {}
for i, target in enumerate(TARGETS):
    ok   = ~np.isnan(Y_va_np[:, i])
    r2   = r2_score(Y_va_np[ok, i], tabm_preds[ok, i])
    rmse = np.sqrt(np.mean((Y_va_np[ok, i] - tabm_preds[ok, i])**2))
    tabm_val_metrics[target] = {'r2': r2, 'rmse': rmse}
    print(f'TabM    {target:<40s}  R²={r2:.4f}  RMSE={rmse:.3f}')


### 7d. TabNet

TabNet (Arik & Pfister, 2019) uses **sequential attention** to select a sparse subset of features
at each decision step — providing both high performance and built-in feature importance.

`pytorch-tabnet` exposes a scikit-learn-compatible `TabNetRegressor`.
One model is trained per target on the same 80/20 location-aware split used by TabPFN and TabM.
Early stopping is handled natively via the `patience` argument.

In [ ]:
from pytorch_tabnet.tab_model import TabNetRegressor

TABNET_MAX_EPOCHS  = 300
TABNET_PATIENCE    = 30
TABNET_BATCH_SIZE  = 512
TABNET_VBATCH_SIZE = 512

# Reuse the same imputed arrays and 80/20 split indices as TabM
X_tn_tr = X_tabm[tr_idx].astype(np.float32)
X_tn_va = X_tabm[va_idx].astype(np.float32)

tabnet_val_metrics = {}

for i, target in enumerate(TARGETS):
    y_tr_tn = Y_tabm[tr_idx, i].reshape(-1, 1).astype(np.float32)
    y_va_tn = Y_tabm[va_idx, i].reshape(-1, 1).astype(np.float32)

    # Drop rows where target is NaN
    ok_tr = ~np.isnan(y_tr_tn[:, 0])
    ok_va = ~np.isnan(y_va_tn[:, 0])

    model_tn = TabNetRegressor(
        n_d=32, n_a=32,
        n_steps=5,
        gamma=1.3,
        n_independent=2, n_shared=2,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-3),
        scheduler_params=dict(step_size=50, gamma=0.9),
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='sparsemax',
        device_name=DEVICE,
        verbose=0,
    )

    model_tn.fit(
        X_tn_tr[ok_tr], y_tr_tn[ok_tr],
        eval_set=[(X_tn_va[ok_va], y_va_tn[ok_va])],
        eval_name=['val'],
        eval_metric=['rmse'],
        max_epochs=TABNET_MAX_EPOCHS,
        patience=TABNET_PATIENCE,
        batch_size=TABNET_BATCH_SIZE,
        virtual_batch_size=TABNET_VBATCH_SIZE,
        drop_last=False,
    )

    preds_va = model_tn.predict(X_tn_va[ok_va])[:, 0]
    true_va  = y_va_tn[ok_va, 0]
    r2   = r2_score(true_va, preds_va)
    rmse = np.sqrt(np.mean((true_va - preds_va) ** 2))
    tabnet_val_metrics[target] = {'r2': r2, 'rmse': rmse, 'model': model_tn}
    print(f'TabNet  {target:<40s}  R²={r2:.4f}  RMSE={rmse:.3f}'
          f'  best_epoch={model_tn.best_epoch}')

### 7e. Four-way comparison (val set)

In [ ]:
comparison_rows = []
for target in TARGETS:
    # CatBoost: mean across K-fold folds
    comparison_rows.append({
        'Target': target, 'Model': 'CatBoost',
        'R²':   np.mean(cv_results[target]['r2']),
        'RMSE': np.mean(cv_results[target]['rmse']),
    })
    # TabPFN, TabM, TabNet: single val-set score
    for name, metrics in [('TabPFN v2', tabpfn_val_metrics),
                           ('TabM k=32', tabm_val_metrics),
                           ('TabNet',    tabnet_val_metrics)]:
        comparison_rows.append({
            'Target': target, 'Model': name,
            'R²':   metrics[target]['r2'],
            'RMSE': metrics[target]['rmse'],
        })

comp_df = (
    pd.DataFrame(comparison_rows)
    .set_index(['Target', 'Model'])
    .round(4)
)

def _highlight_best(s):
    """Bold + green the highest R² within each target group."""
    best = s.max()
    return ['font-weight:bold; color:green' if v == best else '' for v in s]

comp_df.style.apply(_highlight_best, subset=['R²'], level=0)


In [ ]:
# Average importance across all targets and folds
imp_matrix = np.stack(
    [m.get_feature_importance() for t in TARGETS for m in cv_results[t]['models']],
    axis=0
)
feat_rank = (
    pd.Series(imp_matrix.mean(axis=0), index=ALL_FEATURE_COLS)
    .sort_values(ascending=False)
)

top_n = 30
fig, ax = plt.subplots(figsize=(9, top_n * 0.28 + 1))
feat_rank.head(top_n).sort_values().plot.barh(ax=ax, color='steelblue', edgecolor='none')
ax.set_title(f'Top {top_n} features (avg importance across targets & folds)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

# Print features from most important to least important
print('\nFeatures ranked by importance (most to least important):')
for i, (feat, val) in enumerate(feat_rank.items(), 1):
    print(f"{i:2d}. {feat}")


## 8. OOF Predictions vs Actuals

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, target in zip(axes, TARGETS):
    mask   = train_df[target].notna()
    y_true = train_df.loc[mask, target].values
    y_pred = cv_results[target]['oof'][train_df.index[mask]]
    ok     = ~np.isnan(y_pred)
    ax.scatter(y_true[ok], y_pred[ok], alpha=0.3, s=8, edgecolors='none', color='steelblue')
    lo, hi = y_true[ok].min(), y_true[ok].max()
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1)
    ax.set_title(f'{target}\nR²={r2_score(y_true[ok], y_pred[ok]):.3f}')
    ax.set_xlabel('Actual')
    ax.set_ylabel('OOF predicted')
plt.tight_layout()
plt.show()

## 9. Final refit with early stopping\n\nK-fold gives honest metrics. For submission we train one model per target on the\nfull labelled set, using a small 20 % early-stopping validation split to find\nthe right stopping point — no fold averaging needed.

In [ ]:
from sklearn.model_selection import train_test_split

FINAL_VAL_FRAC = 0.20   # fraction held out solely for early-stopping signal
FINAL_OD_WAIT  = 150    # be a bit more patient on the full dataset

final_params = {**_base_params, 'od_wait': FINAL_OD_WAIT}
# Remove a hard iteration cap if the CV avg suggests we need more room
final_params['iterations'] = max(final_params.get('iterations', 2000),
                                 int(max(np.mean(cv_results[t]['best_iters']) for t in TARGETS) * 1.5))

final_models = {}

for target in TARGETS:
    mask   = train_df[target].notna()
    X_all  = train_df.loc[mask, ALL_FEATURE_COLS]
    y_all  = train_df.loc[mask, target]

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_all, y_all,
        test_size=FINAL_VAL_FRAC,
        random_state=SEED,
    )

    model = CatBoostRegressor(**final_params)
    model.fit(
        Pool(X_tr, y_tr),
        eval_set=Pool(X_val, y_val),
    )

    val_preds = model.predict(X_val)
    val_r2    = r2_score(y_val, val_preds)
    print(f'{target}: best_iter={model.best_iteration_}  val R²={val_r2:.4f}')

    final_models[target] = model

## 9. Generate Submission

In [ ]:
X_sub      = sub_df[ALL_FEATURE_COLS]
submission = sub_raw[['Latitude', 'Longitude', 'Sample Date']].copy()

for target in TARGETS:
    submission[target] = np.clip(final_models[target].predict(X_sub), 0, None)

out_path = BASE / 'data/submission_era5_catboost.csv'
submission.to_csv(out_path, index=False)
print('Saved:', out_path)
submission.head(10)